# ⚡ What The Bug — Fine-Tune Gemma 2 for SystemVerilog & UVM Verification

This notebook allows you to fine-tune **Google Gemma 2 (2B / 9B)** on your custom SystemVerilog interview & verification dataset exported from [whathebug.com](https://whathebug.com).

### Pipeline Steps:
1. **Install Dependencies** (Unsloth, PyTorch, Transformers, Datasets, TRL)
2. **Load Base Model** (Gemma 2 2B / 9B in 4-bit quantization)
3. **Load Custom Dataset** (`gemma_dv_fine_tuning_dataset.json` or `gemma_dv_dataset.jsonl` from whathebug.com)
4. **Train LoRA Adapters** (Fast QLoRA fine-tuning)
5. **Evaluate & Compare** (Compare Base Model vs. Fine-Tuned Model on LRM coding problems)

In [ ]:
# Step 1: Install Fast Fine-Tuning Dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl<0.9.0" peft accelerate bitsandbytes datasets transformers

In [ ]:
# Step 2: Load Gemma 2 with 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect: Float16 or Bfloat16
load_in_4bit = True # Use 4bit quantization to fit comfortably in free Colab T4 GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit", # or "unsloth/gemma-2-9b-it-bnb-4bit"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# Step 3: Load Dataset Exported from What The Bug
import json
from datasets import Dataset
from google.colab import files

print("Upload your gemma_dv_fine_tuning_dataset.json or gemma_dv_dataset.jsonl:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, 'r') as f:
    if filename.endswith('.jsonl'):
        data = [json.loads(line) for line in f if line.strip()]
    else:
        data = json.load(f)

print(f"Loaded {len(data)} training examples from {filename}.")

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples.get("input", ["" for _ in instructions])
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = prompt_template.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(data)
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# Step 4: Fine-Tune with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Step 5: Test Fine-Tuned Model Generation
FastLanguageModel.for_inference(model)

test_prompt = prompt_template.format(
    "Write SystemVerilog code for a parameterized synchronous FIFO buffer with full and empty flags.",
    "Topic: SYSTEMVERILOG CODING\nReference: IEEE 1800-2023\nDifficulty: MEDIUM",
    ""
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])